## Section 1 — Setup & Data Loading
Key insight: the OWID vaccinations table gives a daily snapshot for every country-date so I can track rollout speed and coverage progress.
I load the file and focus on Italy so I can compare raw dose flow (daily_vaccinations) with coverage metrics (people_vaccinated_per_hundred).

In [ ]:
# Load the vaccination dataset, filter Italy, and print a quick sanity check.
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path

sns.set_theme(style="whitegrid")
PRIMARY_BLUE = "#003865"
ACCENT_BLUE = "#00A3E0"

data_path = Path("vaccinations.csv")
fallback_url = "https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/vaccinations/vaccinations.csv"

try:
    if data_path.exists():
        vaccinations = pd.read_csv(data_path, parse_dates=["date"])
    else:
        vaccinations = pd.read_csv(fallback_url, parse_dates=["date"])
except Exception as error:
    raise FileNotFoundError("Could not load vaccinations.csv from the workspace or the OWID fallback URL.") from error

italy = vaccinations.loc[vaccinations["location"] == "Italy"].copy().sort_values("date")

print(f"Italy shape: {italy.shape}")
print(italy.head().to_string(index=False))

## Section 2 — Exploratory Analysis

In [ ]:
# Plot Italy's vaccination pace and coverage progress over time.
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharex=True)

sns.lineplot(data=italy, x="date", y="daily_vaccinations", ax=axes[0], color=PRIMARY_BLUE, linewidth=2)
axes[0].set_title("Italy Daily Vaccinations Over Time")
axes[0].set_xlabel("Date")
axes[0].set_ylabel("Daily vaccinations")
axes[0].tick_params(axis="x", rotation=45)

sns.lineplot(data=italy, x="date", y="people_vaccinated_per_hundred", ax=axes[1], color=PRIMARY_BLUE, linewidth=2, label="Italy")
axes[1].axhline(70, color=ACCENT_BLUE, linestyle="--", linewidth=2, label="70% threshold")
axes[1].set_title("Italy People Vaccinated per Hundred")
axes[1].set_xlabel("Date")
axes[1].set_ylabel("People vaccinated per hundred")
axes[1].legend()
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

Key insight: Italy's rollout climbs in clear waves, not as a steady march, which points to bursts of supply or policy-driven pushes.
Those peaks drove coverage higher over time, but Italy needed repeated surges rather than one quick push to approach the 70% mark.
That pattern suggests logistics and outreach mattered as much as dose availability.

## Section 3 — Comparative Benchmarking

In [ ]:
# Compare Italy with peer countries and identify who reached 50% coverage fastest.
peer_countries = ["Italy", "France", "Germany", "Spain", "United Kingdom"]
country_palette = {
    "Italy": PRIMARY_BLUE,
    "France": ACCENT_BLUE,
    "Germany": "#4C78A8",
    "Spain": "#F58518",
    "United Kingdom": "#54A24B",
}

benchmark = vaccinations.loc[vaccinations["location"].isin(peer_countries)].copy().sort_values(["location", "date"])

fig, ax = plt.subplots(figsize=(14, 7))
sns.lineplot(data=benchmark, x="date", y="people_vaccinated_per_hundred", hue="location", palette=country_palette, linewidth=2, ax=ax)
ax.set_title("People Vaccinated per Hundred Across Five European Countries")
ax.set_xlabel("Date")
ax.set_ylabel("People vaccinated per hundred")
ax.legend(title="Country")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

coverage_start = benchmark.dropna(subset=["people_vaccinated_per_hundred"]).groupby("location")["date"].min()
coverage_50 = benchmark.loc[benchmark["people_vaccinated_per_hundred"] >= 50].groupby("location")["date"].min()
days_to_50 = (coverage_50 - coverage_start).dt.days.dropna()
fastest_country = days_to_50.idxmin()
fastest_days = int(days_to_50.min())
print(f"{fastest_country} reached 50% coverage fastest in {fastest_days} days.")

Key insight: some countries hit 50% way faster than others, and that speed tells us whether they had better logistics or stronger demand generation.
I compare Italy, France, Germany, Spain and the UK so we can see who moved fastest and who might need more operational support instead of just more doses.
That matters because sending doses to a place that can't deliver them quickly wastes time — operational fixes often beat raw supply.

## Section 4 — Priority Gap Analysis

In [ ]:
# Compute Italy's coverage gap score and visualize how it changes over time.
italy_gap = italy.loc[:, ["date", "people_vaccinated_per_hundred"]].copy()
italy_gap["coverage_gap_score"] = (70 - italy_gap["people_vaccinated_per_hundred"]).clip(lower=0)

fig, ax = plt.subplots(figsize=(12, 6))
ax.fill_between(italy_gap["date"], italy_gap["coverage_gap_score"], color=PRIMARY_BLUE, alpha=0.25, label="Coverage gap score")
sns.lineplot(data=italy_gap, x="date", y="coverage_gap_score", ax=ax, color=PRIMARY_BLUE, linewidth=2)
ax.set_title("Italy Coverage Gap Score Over Time")
ax.set_xlabel("Date")
ax.set_ylabel("Coverage gap score")
ax.legend()
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

Key insight: the coverage gap score (70 − vaccinated per hundred) turns the remaining task into a single daily number I can track.
When that score stays high, I’d push extra doses and staffing to those dates and places; when it drops, I’d shift to targeted outreach and equity work.
The score gives a simple way to prioritize where extra effort will close the most percentage points.

## Section 5 — Decision Framework

In [ ]:
# Build a country summary table and compare peak daily vaccination intensity.
def daily_vaccinations_per_hundred(country_frame: pd.DataFrame) -> pd.Series:
    if "daily_vaccinations_per_hundred" in country_frame.columns:
        return country_frame["daily_vaccinations_per_hundred"]
    if "daily_vaccinations_per_million" in country_frame.columns:
        return country_frame["daily_vaccinations_per_million"] / 10
    if "population" in country_frame.columns and country_frame["population"].notna().any():
        return country_frame["daily_vaccinations"] / country_frame["population"] * 100
    return pd.Series(np.nan, index=country_frame.index)

def build_country_summary(country_frame: pd.DataFrame) -> pd.Series:
    ordered = country_frame.sort_values("date").copy()
    daily_series = ordered["daily_vaccinations"]
    coverage_series = ordered["people_vaccinated_per_hundred"].dropna()
    peak_idx = daily_series.idxmax() if daily_series.notna().any() else np.nan
    peak_daily_vaccinations = ordered.loc[peak_idx, "daily_vaccinations"] if pd.notna(peak_idx) else np.nan
    peak_date = ordered.loc[peak_idx, "date"] if pd.notna(peak_idx) else pd.NaT
    first_date = ordered["date"].min()
    date_50 = ordered.loc[ordered["people_vaccinated_per_hundred"] >= 50, "date"].min()
    days_to_50 = (date_50 - first_date).days if pd.notna(date_50) else np.nan
    final_coverage = coverage_series.iloc[-1] if not coverage_series.empty else np.nan
    peak_daily_per_hundred = daily_vaccinations_per_hundred(ordered).max()
    return pd.Series({
        "Peak daily vaccinations": peak_daily_vaccinations,
        "Date of peak": peak_date,
        "Days to reach 50% coverage": days_to_50,
        "Final coverage %": final_coverage,
        "Peak daily vaccinations per hundred": peak_daily_per_hundred,
    })

summary_df = benchmark.groupby("location").apply(build_country_summary).reset_index()
print(summary_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=summary_df, x="location", y="Peak daily vaccinations per hundred", palette=country_palette, ax=ax)
ax.set_title("Peak Daily Vaccinations per Hundred by Country")
ax.set_xlabel("Country")
ax.set_ylabel("Peak daily vaccinations per hundred")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

Key insight: prioritize extra doses and operational support to places with the largest coverage gaps instead of splitting supply evenly.
Boost clinic hours, mobile units, and local outreach where the gap stays large so doses actually turn into vaccinated people quickly.
Limitation: this analysis uses national averages, so I can’t see local pockets of low coverage.